# HEALPix Aggregate

> Aggregate data by HEALPix cells (batch processing)

In [ ]:
#| default_exp aggregate

In [ ]:
#| export
#| eval: false
"""
Aggregate parquet data by HEALPix cells using sidecar mapping files.

This script implements the "apply" step of a split-apply-combine strategy for
processing large spatial datasets with HEALPix tessellation.

The workflow:
1. Split: Generate HEALPix sidecar files (source_id -> healpix_id mapping)
2. Apply: Aggregate data by HEALPix cells (this script)
3. Combine: Merge aggregated results

Sidecar files encode metadata in their filenames using dot-separated segments:
  Example: input.assignment-strict.healpix_nside-128_order-nested.parquet
  
  Parsed as:
    - assignment: strict (mode)
    - nside: 128
    - order: nested
"""

import argparse
import logging
import sys
import os
from pathlib import Path
from typing import Optional, Sequence, Iterable, Callable, Dict, List
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import re

try:
    from tqdm.auto import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False
    # Fallback: no-op progress bar
    class tqdm:
        def __init__(self, iterable, *args, **kwargs):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable)
        def set_postfix(self, *args, **kwargs):
            pass
        def close(self):
            pass

try:
    import dask.dataframe as dd
    DASK_AVAILABLE = True
except ImportError:
    DASK_AVAILABLE = False
    dd = None

try:
    import duckdb
    DUCKDB_AVAILABLE = True
except ImportError:
    DUCKDB_AVAILABLE = False
    duckdb = None


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# Aggregation function lookup
def _mad(arr: np.ndarray) -> float:
    """Compute Median Absolute Deviation."""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.median(np.abs(arr - np.median(arr))))


def _robust_std(arr: np.ndarray) -> float:
    """Compute robust standard deviation (MAD * 1.4826)."""
    m = _mad(arr)
    if np.isnan(m):
        return float("nan")
    return float(m * 1.4826)  # Approximation for normal distribution


AGG_LOOKUP: Dict[str, Callable] = {
    "mean": lambda a: float(np.nanmean(a)) if np.any(np.isfinite(a)) else float("nan"),
    "median": lambda a: float(np.nanmedian(a)) if np.any(np.isfinite(a)) else float("nan"),
    "std": lambda a: float(np.nanstd(a, ddof=0)) if np.any(np.isfinite(a)) else float("nan"),
    "min": lambda a: float(np.nanmin(a)) if np.any(np.isfinite(a)) else float("nan"),
    "max": lambda a: float(np.nanmax(a)) if np.any(np.isfinite(a)) else float("nan"),
    "mad": lambda a: _mad(a),
    "robust_std": lambda a: _robust_std(a),
}


def generate_output_filename(
    input_file: Path,
    sidecar_file: Path,
    output_dir: Optional[Path] = None
) -> Path:
    """
    Generate output filename that matches the parseable structure of the sidecar.
    
    Pattern: <stem>-aggregated.<sidecar_suffix>.parquet
    
    Args:
        input_file: Original input parquet file
        sidecar_file: Sidecar file being used
        output_dir: Optional output directory (default: same as sidecar_file)
        
    Returns:
        Path to output file
        
    Example:
        input: mascs_data_MeSS.parquet
        sidecar: mascs_data_MeSS.cell-healpix_assignment-strict_nside-64_order-nested.parquet
        output: mascs_data_MeSS-aggregated.cell-healpix_assignment-strict_nside-64_order-nested.parquet
    """
    input_stem = input_file.stem
    sidecar_name = sidecar_file.name
    
    # Extract the suffix after input_stem
    if sidecar_name.startswith(f"{input_stem}."):
        # Get everything after "input_stem."
        sidecar_suffix = sidecar_name[len(input_stem) + 1:]
        # Remove .parquet extension
        if sidecar_suffix.endswith('.parquet'):
            sidecar_suffix = sidecar_suffix[:-len('.parquet')]
    else:
        # Fallback: use whole sidecar name without extension
        logger.warning(f"Sidecar name doesn't start with expected stem '{input_stem}'")
        sidecar_suffix = sidecar_file.stem
    
    # Build output filename
    output_name = f"{input_stem}-aggregated.{sidecar_suffix}.parquet"
    
    # Use output_dir if specified, otherwise use sidecar directory
    if output_dir is None:
        output_dir = sidecar_file.parent
    
    return output_dir / output_name


def collect_sidecar_outputs(
    input_parquet: Path,
    output_dir: Path,
    read_stats: bool = False
) -> pd.DataFrame:
    """
    Scan output_dir for sidecar parquet files that match the input file stem.
    
    Parses metadata from filenames with dot-separated segments and underscore-separated
    key-value pairs (key-value format).
    
    Args:
        input_parquet: Path to the original parquet file
        output_dir: Directory containing sidecar files
        read_stats: If True, read files to compute row counts and unique healpix counts
        
    Returns:
        DataFrame with columns: file, coalesced, mode, nside, order, n_rows, n_unique_healpix
    """
    stem = input_parquet.stem
    
    if not output_dir.exists():
        raise FileNotFoundError(f"Output directory does not exist: {output_dir}")
    
    logger.debug(f"Scanning {output_dir} for sidecar files matching stem: {stem}")
    
    rows = []
    for p in output_dir.rglob("*.parquet"):
        # Skip partition-directory style outputs
        if p.name.endswith(".parts"):
            continue
        if not p.is_file():
            continue
        
        name = p.name
        # Ensure file starts with expected stem + '.'
        if not name.startswith(f"{stem}."):
            continue
        
        logger.debug(f"Found sidecar candidate: {name}")
        
        # Strip trailing ".parquet"
        if name.lower().endswith(".parquet"):
            base = name[:-len(".parquet")]
        else:
            base = name
        
        # Tail after the first dot
        tail = base[len(stem) + 1:] if len(base) > len(stem) else ""
        if not tail:
            continue
        
        meta = {}
        # Split by '.' then groups by '_' and key/val by '-'
        for seg in tail.split("."):
            for group in seg.split("_"):
                if "-" not in group:
                    continue
                k, v = group.split("-", 1)
                if not k:
                    continue
                meta[k] = v
        
        # Normalize expected keys
        out_row = {"file": str(p), "coalesced": True}
        
        # Map 'assignment' -> 'mode' for compatibility
        if "assignment" in meta and "mode" not in meta:
            out_row["mode"] = meta.pop("assignment")
        
        # Copy remaining meta
        for k, v in meta.items():
            out_row[k] = v
        
        # Cast nside to int if present
        if "nside" in out_row:
            try:
                out_row["nside"] = int(out_row["nside"])
            except Exception:
                pass
        
        # Optional lightweight stats
        if read_stats:
            n_rows = None
            n_unique = None
            try:
                df = pd.read_parquet(p, columns=["source_id", "healpix_id"])
                n_rows = int(len(df))
                if "healpix_id" in df.columns:
                    n_unique = int(df["healpix_id"].nunique())
            except Exception as e:
                logger.warning(f"Could not read stats from {p.name}: {e}")
                n_rows = None
                n_unique = None
            out_row["n_rows"] = n_rows
            out_row["n_unique_healpix"] = n_unique
        
        rows.append(out_row)
    
    if not rows:
        logger.warning(f"No sidecar files found matching stem: {stem}")
        return pd.DataFrame(
            columns=[
                "file",
                "coalesced",
                "mode",
                "nside",
                "order",
                "n_rows",
                "n_unique_healpix",
            ]
        )
    
    df_out = pd.DataFrame(rows)
    
    # Ensure nside/int dtypes where possible
    if "nside" in df_out.columns:
        try:
            df_out["nside"] = pd.to_numeric(df_out["nside"], errors="coerce").astype("Int64")
        except Exception:
            pass
    
    # Sort by nside then mode if available
    sort_keys = [k for k in ("nside", "mode") if k in df_out.columns]
    if sort_keys:
        df_out = df_out.sort_values(sort_keys).reset_index(drop=True)
    else:
        df_out = df_out.reset_index(drop=True)
    
    logger.info(f"Found {len(df_out)} sidecar file(s)")
    return df_out


def print_parquet_schema(file_path: Path, show_metadata: bool = True) -> None:
    """
    Print the schema of a parquet file.
    
    Args:
        file_path: Path to the parquet file
        show_metadata: If True, also print file-level metadata
    """
    try:
        # Read metadata without reading data
        parquet_file = pq.ParquetFile(file_path)
        schema = parquet_file.schema_arrow
        
        print(f"\n{'='*70}")
        print(f"Schema for: {file_path.name}")
        print(f"{'='*70}")
        
        # Print field information
        print(f"\nFields ({len(schema)}):")
        for field in schema:
            print(f"  {field.name:30s} {str(field.type):20s}")
        
        # Print metadata if present
        if show_metadata and schema.metadata:
            print("\nMetadata:")
            for key, value in schema.metadata.items():
                key_str = key.decode() if isinstance(key, bytes) else key
                val_str = value.decode() if isinstance(value, bytes) else value
                print(f"  {key_str}: {val_str}")
        
        # Get file stats
        print(f"\nFile statistics:")
        print(f"  Number of row groups: {parquet_file.num_row_groups}")
        print(f"  Total rows: {parquet_file.metadata.num_rows}")
        print(f"  Compressed size: {parquet_file.metadata.serialized_size / 1024 / 1024:.2f} MB")
        
        print(f"{'='*70}\n")
        
    except Exception as e:
        logger.error(f"Error reading schema from {file_path}: {e}")
        sys.exit(1)


def print_sidecar_summary(sidecars_df: pd.DataFrame, input_file: Path) -> None:
    """
    Print a formatted summary of available sidecar files as an ASCII table.
    
    Args:
        sidecars_df: DataFrame from collect_sidecar_outputs
        input_file: Original input parquet file
    """
    print(f"\n{'='*70}")
    print(f"Input file: {input_file.name}")
    print(f"{'='*70}\n")
    
    if len(sidecars_df) == 0:
        print("No sidecar files found.")
        print(f"{'='*70}\n")
        return
    
    print(f"Found {len(sidecars_df)} sidecar file(s):\n")
    
    # Create display dataframe with selected columns
    display_df = sidecars_df.copy()
    
    # Add index column for reference
    display_df.insert(0, 'idx', range(len(display_df)))
    
    # Select and order columns for display
    display_cols = ['idx']
    
    # Always include these if present
    priority_cols = ['mode', 'nside', 'order']
    for col in priority_cols:
        if col in display_df.columns:
            display_cols.append(col)
    
    # Add statistics columns if present
    stat_cols = ['n_rows', 'n_unique_healpix']
    for col in stat_cols:
        if col in display_df.columns:
            display_cols.append(col)
    
    # Add any other metadata columns that aren't 'file' or 'coalesced'
    other_cols = [c for c in display_df.columns 
                  if c not in display_cols + ['file', 'coalesced']]
    display_cols.extend(other_cols)
    
    # Keep only existing columns
    display_cols = [c for c in display_cols if c in display_df.columns]
    
    # Create final display dataframe
    display_df = display_df[display_cols]
    
    # Format the table using pandas to_string with nice formatting
    table_str = display_df.to_string(index=False, na_rep='', max_colwidth=50)
    print(table_str)
    
    # Print filename mapping below the table
    print(f"\n{'Filenames:':-<70}")
    for idx, row in sidecars_df.iterrows():
        filename = Path(row['file']).name
        print(f"  [{idx}] {filename}")
    
    print(f"{'='*70}\n")


def densify_healpix_aggregates(
    agg_sparse_df: pd.DataFrame,
    nside: int,
    index_name: str | None = None
) -> pd.DataFrame:
    """
    Return a densified DataFrame with one row per HEALPix cell for given nside.
    
    Args:
        agg_sparse_df: Output of aggregate_by_sidecar indexed by healpix_id
        nside: HEALPix nside used to compute full pixel count (npix = 12 * nside^2)
        index_name: Optional index name to use (defaults to agg_sparse_df.index.name or 'healpix_id')
        
    Returns:
        DataFrame with full HEALPix grid. For newly added cells:
        - float columns get np.nan
        - integer columns get pandas nullable Int64 dtype with pd.NA
    """
    if index_name is None:
        index_name = agg_sparse_df.index.name or "healpix_id"
    
    npix = 12 * int(nside) ** 2
    full_index = pd.Index(range(npix), name=index_name)
    
    logger.debug(f"Densifying from {len(agg_sparse_df)} cells to {npix} cells")
    
    # Ensure frame has index name matching requested
    df = agg_sparse_df.copy()
    if df.index.name != index_name:
        df.index = df.index.astype("int64")
        df.index.name = index_name
    
    # Detect original dtypes before reindexing
    orig_dtypes = df.dtypes.copy()
    int_like_cols = [c for c, dt in orig_dtypes.items() if pd.api.types.is_integer_dtype(dt)]
    float_like_cols = [c for c, dt in orig_dtypes.items() if pd.api.types.is_float_dtype(dt)]
    
    # Reindex (missing rows created)
    df = df.reindex(full_index)
    
    # Restore float columns dtype and ensure NaN in missing
    for c in float_like_cols:
        if c in df.columns:
            df[c] = df[c].astype("float64")
    
    # Convert integer-like columns to pandas nullable Int64 so missing become <NA>
    for c in int_like_cols:
        if c in df.columns:
            df[c] = df[c].astype("Int64")
    
    logger.info(f"Densified to {len(df)} cells (added {len(df) - len(agg_sparse_df)} empty cells)")
    return df


def aggregate_by_sidecar(
    original: pd.DataFrame,
    sidecar: pd.DataFrame,
    value_columns: list[str],
    aggs: list[str] | None = None,
    source_id_col: str = "source_id",
    healpix_col: str = "healpix_id",
    min_count: int = 0,
    include_counts: bool = True,
    sentinel_threshold: float = 1e30,
) -> pd.DataFrame:
    """
    Aggregate columns from original DataFrame according to sidecar mapping.
    
    Args:
        original: Original DataFrame with data to aggregate
        sidecar: Sidecar DataFrame with source_id -> healpix_id mapping
        value_columns: List of column names to aggregate
        aggs: List of aggregation function names (default: mean, median, std, mad, robust_std)
        source_id_col: Name of source ID column
        healpix_col: Name of HEALPix ID column
        min_count: Minimum number of sources required per cell
        include_counts: Include n_sources and n_sources_used columns
        sentinel_threshold: Values >= this are masked as NaN (e.g., 1e32 saturation values)
        
    Returns:
        DataFrame indexed by healpix_id with aggregated columns
    """
    if aggs is None:
        aggs = ["mean", "median", "std", "robust_std"]
    
    logger.info(f"Starting aggregation for {len(value_columns)} column(s): {value_columns}")
    logger.info(f"Aggregation functions: {aggs}")
    logger.debug(f"Min count threshold: {min_count}, Sentinel threshold: {sentinel_threshold}")
    
    # Validate inputs
    missing = [c for c in value_columns if c not in original.columns]
    if missing:
        raise KeyError(f"Value columns not found in original dataframe: {missing}")
    
    unsupported = [a for a in aggs if a not in AGG_LOOKUP]
    if unsupported:
        raise ValueError(f"Unsupported aggregations: {unsupported}. Available: {list(AGG_LOOKUP.keys())}")
    
    if source_id_col not in sidecar.columns or healpix_col not in sidecar.columns:
        raise KeyError(f"Sidecar must contain '{source_id_col}' and '{healpix_col}' columns")
    
    # Ensure sidecar source_id dtype is int64
    logger.debug("Preparing sidecar source_id column")
    sidecar = sidecar.copy()
    try:
        sidecar[source_id_col] = sidecar[source_id_col].astype("int64")
    except Exception:
        sidecar[source_id_col] = pd.to_numeric(sidecar[source_id_col], errors="coerce").astype("Int64").astype("int64")
    
    # Prepare original DataFrame
    logger.debug("Preparing original DataFrame")
    orig = original.copy()
    
    # Create source_id if not present (use index)
    if source_id_col not in orig.columns:
        logger.info(f"Creating {source_id_col} column from DataFrame index")
        orig = orig.reset_index().rename(columns={"index": source_id_col})
    
    # Coerce original source_id dtype to int64
    try:
        orig[source_id_col] = orig[source_id_col].astype("int64")
    except Exception:
        orig[source_id_col] = pd.to_numeric(orig[source_id_col], errors="coerce").astype("Int64").astype("int64")
    
    # Check for duplicates
    if orig[source_id_col].duplicated().any():
        dup_count = int(orig[source_id_col].duplicated().sum())
        logger.warning(f"Found {dup_count} duplicate source_id values - keeping first occurrence")
        orig = orig.drop_duplicates(subset=source_id_col, keep="first")
    
    # Coerce value columns to numeric and mask sentinel values
    logger.debug("Processing value columns")
    for col in value_columns:
        orig[col] = pd.to_numeric(orig[col], errors="coerce")
        
        # Mask extreme sentinel values
        try:
            mask_big = orig[col].abs() >= float(sentinel_threshold)
        except Exception:
            mask_big = pd.Series(False, index=orig.index)
        
        if mask_big.any():
            n_masked = int(mask_big.sum())
            logger.info(f"Masking {n_masked} sentinel values in column '{col}' (>= {sentinel_threshold})")
            orig.loc[mask_big, col] = np.nan
    
    # Keep only necessary columns
    cols_to_keep = [source_id_col] + list(value_columns)
    orig = orig[cols_to_keep]
    
    # Diagnostic: check overlap between sidecar and original
    side_ids = pd.Index(sidecar[source_id_col].unique())
    orig_ids = pd.Index(orig[source_id_col].unique())
    inter = side_ids.intersection(orig_ids)
    
    if len(side_ids) == 0:
        logger.error("Sidecar contains 0 source_ids")
        raise ValueError("Sidecar is empty")
    
    pct = 100.0 * len(inter) / len(side_ids)
    logger.info(f"Sidecar source_id overlap: {len(inter)}/{len(side_ids)} ({pct:.1f}%)")
    
    if len(inter) < len(side_ids):
        n_missing = len(side_ids) - len(inter)
        logger.warning(f"{n_missing} source_ids in sidecar not found in original data")
        if logger.level <= logging.DEBUG:
            missing_sample = list(side_ids.difference(orig_ids)[:10])
            logger.debug(f"Sample missing source_ids: {missing_sample}")
    
    # Merge sidecar with original data
    logger.info("Merging sidecar with original data")
    merged = sidecar[[source_id_col, healpix_col]].merge(
        orig, on=source_id_col, how="left"
    )
    logger.debug(f"Merged dataframe shape: {merged.shape}")
    
    # Group by healpix_id and aggregate
    logger.info(f"Grouping by {healpix_col} and computing aggregations")
    rows: list[dict] = []
    grouped = merged.groupby(healpix_col, sort=True)
    n_groups = len(grouped)
    logger.info(f"Processing {n_groups} HEALPix cells")
    
    # Create progress bar
    pbar = tqdm(
        grouped,
        total=n_groups,
        desc="Aggregating HEALPix cells",
        unit="cell",
        disable=(logger.level > logging.INFO)  # Disable if quiet mode
    )
    
    for hp_value, grp in pbar:
        row: dict = {healpix_col: hp_value}
        n_sources = int(len(grp))
        
        # Apply min_count threshold
        if n_sources < int(min_count):
            logger.debug(f"Cell {hp_value}: {n_sources} sources < min_count={min_count}, setting to NaN")
            for col in value_columns:
                for agg_name in aggs:
                    row[f"{col}_{agg_name}"] = np.nan
            if include_coun//ts:
                row["n_sources"] = n_sources
            rows.append(row)
            continue
        
        # Compute aggregates
        for col in value_columns:
            arr = grp[col].dropna().to_numpy()
            n_used = arr.size
            
            if n_used == 0:
                for agg_name in aggs:
                    row[f"{col}_{agg_name}"] = np.nan
            else:
                for agg_name in aggs:
                    func = AGG_LOOKUP[agg_name]
                    try:
                        row[f"{col}_{agg_name}"] = float(func(arr))
                    except Exception as e:
                        logger.debug(f"Error computing {agg_name} for {col} in cell {hp_value}: {e}")
                        row[f"{col}_{agg_name}"] = np.nan
        
        if include_counts:
            row["n_sources"] = n_sources
        
        rows.append(row)
        
        # Update progress bar postfix with cell info every 100 cells
        if len(rows) % 100 == 0:
            pbar.set_postfix({
                'cell_id': hp_value,
                'n_src': n_sources,
                'processed': len(rows)
            })
    
    pbar.close()
    
    if not rows:
        logger.warning("No aggregation results - returning empty DataFrame")
        cols = []
        for col in value_columns:
            for agg_name in aggs:
                cols.append(f"{col}_{agg_name}")
        if include_counts:
            cols += ["n_sources"]
        return pd.DataFrame(columns=cols).set_index(pd.Index([], name=healpix_col))
    
    logger.info(f"Creating output DataFrame with {len(rows)} cells")
    df_out = pd.DataFrame(rows).set_index(healpix_col).sort_index()
    
    # Ensure proper dtypes
    for col in df_out.columns:
        if any(col.endswith(f"_{a}") for a in aggs):
            df_out[col] = df_out[col].astype("float64")
    
    # Ensure n_sources is int64
    if "n_sources" in df_out.columns:
        df_out["n_sources"] = df_out["n_sources"].astype("int64")
    
    logger.info(f"Aggregation complete. Output shape: {df_out.shape}")
    return df_out


def print_dry_run_summary(
    input_file: Path,
    sidecar_path: Path,
    output_path: Path,
    columns: list[str],
    aggs: list[str],
    filter_expr: Optional[str],
    min_count: int,
    densify: bool,
    use_duckdb: bool,
    use_dask: bool,
    dask_npartitions: Optional[int]
) -> None:
    """
    Print a summary of what would be done without --dry-run.
    
    Args:
        input_file: Input parquet file
        sidecar_path: Sidecar file to use
        output_path: Output file path
        columns: Columns to aggregate
        aggs: Aggregation functions
        filter_expr: Filter expression
        min_count: Minimum count threshold
        densify: Whether to densify output
        use_duckdb: Whether to use DuckDB
        use_dask: Whether to use Dask
        dask_npartitions: Number of Dask partitions
    """
    print(f"\n{'='*70}")
    print("DRY RUN - No files will be modified")
    print(f"{'='*70}\n")
    
    print("Configuration:")
    print(f"  Input file:       {input_file}")
    print(f"  Sidecar file:     {sidecar_path.name}")
    print(f"  Output file:      {output_path}")
    print()
    
    print("Processing options:")
    print(f"  Columns:          {', '.join(columns)}")
    print(f"  Aggregations:     {', '.join(aggs)}")
    print(f"  Filter:           {filter_expr or 'None'}")
    print(f"  Min count:        {min_count}")
    print(f"  Densify:          {densify}")
    print()
    
    print("Performance options:")
    print(f"  Use DuckDB:       {use_duckdb}")
    print(f"  Use Dask:         {use_dask}")
    if use_dask:
        print(f"  Dask partitions:  {dask_npartitions or 'auto'}")
    print()
    
    print("Actions that would be performed:")
    print("  1. Load sidecar file")
    print(f"  2. Load input data ({len(columns)} column(s))")
    if filter_expr:
        print(f"  3. Apply filter: {filter_expr}")
    print("  4. Merge with sidecar mapping")
    print("  5. Aggregate by HEALPix cells")
    if densify:
        print("  6. Densify to full HEALPix grid")
    print(f"  7. Write output to: {output_path}")
    print(f"  8. Write metadata to: {output_path.with_suffix('.meta.json')}")
    
    print(f"\n{'='*70}")
    print("To execute, run the same command without --dry-run or -n")
    print(f"{'='*70}\n")


def main():
    """Main entry point for the CLI script."""
    parser = argparse.ArgumentParser(
        description="Aggregate parquet data by HEALPix cells using sidecar mapping files.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # List available sidecars
  %(prog)s data.parquet -d output/ --list-sidecars
  
  # List sidecars with statistics
  %(prog)s data.parquet -d output/ -l -s
  
  # Show input file schema
  %(prog)s data.parquet --schema
  
  # Aggregate using a specific sidecar
  %(prog)s data.parquet -d output/ --sidecar-index 0 --aggregate
        """
    )
    
    # Positional arguments
    parser.add_argument(
        'input',
        type=Path,
        help='Input parquet file to process'
    )
    
    # Sidecar directory
    parser.add_argument(
        '-d', '--sidecar-dir',
        type=Path,
        default=Path('output'),
        metavar='DIR',
        help='Directory containing sidecar files (default: output/)'
    )
    
    # Information display options
    parser.add_argument(
        '-l', '--list-sidecars',
        action='store_true',
        help='List all available sidecar files for the input'
    )
    
    parser.add_argument(
        '-s', '--stats',
        action='store_true',
        help='Include statistics (row counts, unique healpix) when listing sidecars'
    )
    
    parser.add_argument(
        '--schema',
        action='store_true',
        help='Print the schema of the input parquet file'
    )
    
    parser.add_argument(
        '--sidecar-schema',
        type=int,
        metavar='INDEX',
        help='Print the schema of a specific sidecar file by index'
    )
    
    # Aggregation options
    parser.add_argument(
        '--sidecar-index',
        type=int,
        metavar='INDEX',
        help='Index of the sidecar file to use for aggregation'
    )
    
    parser.add_argument(
        '--aggregate',
        action='store_true',
        help='Perform aggregation'
    )
    
    parser.add_argument(
        '--columns',
        nargs='+',
        metavar='COL',
        help='Column names to aggregate (required for --aggregate)'
    )
    
    parser.add_argument(
        '--aggs',
        nargs='+',
        metavar='AGG',
        choices=list(AGG_LOOKUP.keys()),
        help=f"Aggregation functions (choices: {', '.join(AGG_LOOKUP.keys())}). Default: mean median std robust_std"
    )
    
    parser.add_argument(
        '--filter',
        type=str,
        metavar='EXPR',
        help="Pandas query expression to filter data (e.g., '(b != 3) and (c != 3)')"
    )
    
    parser.add_argument(
        '--min-count',
        type=int,
        default=0,
        metavar='N',
        help='Minimum sources per cell (default: 0)'
    )
    
    parser.add_argument(
        '--densify',
        action='store_true',
        help='Densify output to include all HEALPix cells (fills empty cells with NaN/NA)'
    )
    
    parser.add_argument(
        '--use-duckdb',
        action='store_true',
        default=True,
        help='Use DuckDB for efficient column selection and filtering (default: True)'
    )
    
    parser.add_argument(
        '--no-duckdb',
        action='store_true',
        help='Disable DuckDB and use pandas/dask for loading'
    )
    
    parser.add_argument(
        '--use-dask',
        action='store_true',
        help='Use Dask for parallel processing of aggregation (works with DuckDB)'
    )
    
    parser.add_argument(
        '--dask-npartitions',
        type=int,
        metavar='N',
        help='Number of Dask partitions (default: auto-detect from CPU count)'
    )
    
    parser.add_argument(
        '-o', '--output',
        type=Path,
        nargs='?',
        const=None,
        default=None,
        metavar='FILE',
        help='Output parquet file or directory (default: auto-generated from sidecar)'
    )
    
    parser.add_argument(
        '-n', '--dry-run',
        action='store_true',
        help='Show what would be done without executing'
    )
    
    # Logging options
    parser.add_argument(
        '-v', '--verbose',
        action='store_true',
        help='Enable verbose (debug) logging'
    )
    
    parser.add_argument(
        '-q', '--quiet',
        action='store_true',
        help='Suppress all output except errors'
    )
    
    args = parser.parse_args()
    
    # Configure logging level
    if args.quiet:
        logger.setLevel(logging.ERROR)
    elif args.verbose:
        logger.setLevel(logging.DEBUG)
    
    # Validate input file
    if not args.input.exists():
        logger.error(f"Input file not found: {args.input}")
        sys.exit(1)
    
    if not args.input.is_file():
        logger.error(f"Not a file: {args.input}")
        sys.exit(1)
    
    # Show schema if requested
    if args.schema:
        print_parquet_schema(args.input)
    
    # List sidecars if requested
    if args.list_sidecars or args.sidecar_index is not None or args.sidecar_schema is not None:
        if not args.sidecar_dir.exists():
            logger.error(f"Sidecar directory not found: {args.sidecar_dir}")
            sys.exit(1)
        
        sidecars_df = collect_sidecar_outputs(
            args.input,
            args.sidecar_dir,
            read_stats=args.stats
        )
        
        if args.list_sidecars:
            print_sidecar_summary(sidecars_df, args.input)
        
        # Show sidecar schema if requested
        if args.sidecar_schema is not None:
            if args.sidecar_schema < 0 or args.sidecar_schema >= len(sidecars_df):
                logger.error(f"Invalid sidecar index: {args.sidecar_schema}. Valid range: 0-{len(sidecars_df)-1}")
                sys.exit(1)
            sidecar_path = Path(sidecars_df.iloc[args.sidecar_schema]['file'])
            print_parquet_schema(sidecar_path)
    
    # Aggregation
    if args.aggregate:
        # Validate required arguments
        if args.sidecar_index is None:
            logger.error("--sidecar-index is required when using --aggregate")
            sys.exit(1)
        
        if not args.columns:
            logger.error("--columns is required when using --aggregate")
            sys.exit(1)
        
        # Get sidecars
        if not args.sidecar_dir.exists():
            logger.error(f"Sidecar directory not found: {args.sidecar_dir}")
            sys.exit(1)
        
        sidecars_df = collect_sidecar_outputs(args.input, args.sidecar_dir, read_stats=False)
        
        if args.sidecar_index < 0 or args.sidecar_index >= len(sidecars_df):
            logger.error(f"Invalid sidecar index: {args.sidecar_index}. Valid range: 0-{len(sidecars_df)-1}")
            sys.exit(1)
        
        sidecar_path = Path(sidecars_df.iloc[args.sidecar_index]['file'])
        logger.info(f"Using sidecar: {sidecar_path.name}")
        
        # Handle output path: if not provided, generate from sidecar structure
        if args.output is None:
            # Use sidecar_dir if specified, otherwise use current directory
            output_dir = args.sidecar_dir if args.sidecar_dir.exists() else Path.cwd()
            args.output = generate_output_filename(args.input, sidecar_path, output_dir)
            logger.info(f"Generated output filename: {args.output}")
        elif args.output.is_dir():
            # If output is a directory, generate filename there
            args.output = generate_output_filename(args.input, sidecar_path, args.output)
            logger.info(f"Output is directory, using: {args.output}")
        elif args.output.suffix == '':
            # If output has no extension, treat as directory and create it
            args.output.mkdir(parents=True, exist_ok=True)
            args.output = generate_output_filename(args.input, sidecar_path, args.output)
            logger.info(f"Output has no extension (directory), using: {args.output}")
        
        # Dry run: show what would be done and exit
        if args.dry_run:
            aggs_to_use = args.aggs or ['mean', 'median', 'std', 'robust_std']
            use_duckdb_check = args.use_duckdb and not args.no_duckdb and DUCKDB_AVAILABLE
            use_dask_check = args.use_dask and DASK_AVAILABLE
            
            print_dry_run_summary(
                input_file=args.input,
                sidecar_path=sidecar_path,
                output_path=args.output,
                columns=args.columns,
                aggs=aggs_to_use,
                filter_expr=args.filter,
                min_count=args.min_count,
                densify=args.densify,
                use_duckdb=use_duckdb_check,
                use_dask=use_dask_check,
                dask_npartitions=args.dask_npartitions
            )
            sys.exit(0)
        
        # Determine whether to use DuckDB
        use_duckdb = args.use_duckdb and not args.no_duckdb
        
        # Check availability
        if use_duckdb and not DUCKDB_AVAILABLE:
            logger.warning("DuckDB not available. Install with: pip install duckdb")
            logger.warning("Falling back to pandas/dask")
            use_duckdb = False
        
        if args.use_dask and not DASK_AVAILABLE:
            logger.warning("Dask not available. Install with: pip install dask[dataframe]")
            logger.warning("Disabling Dask")
            args.use_dask = False
        
        # Determine columns to load (only what we need)
        cols_needed = list(args.columns)  # value columns to aggregate
        
        # Check if source_id exists in the file
        source_id_col = 'source_id'
        try:
            pf = pq.ParquetFile(args.input)
            available_cols = pf.schema_arrow.names
            
            # Add source_id if it exists, otherwise we'll use index
            if source_id_col in available_cols:
                if source_id_col not in cols_needed:
                    cols_needed = [source_id_col] + cols_needed
            else:
                logger.info(f"Column '{source_id_col}' not found - will use DataFrame index")
        except Exception as e:
            logger.warning(f"Could not read parquet schema: {e}")
        
        # Load data
        if use_duckdb:
            logger.info(f"Loading data with DuckDB (efficient column selection)")
            logger.info(f"Columns to load: {cols_needed}")
            
            try:
                # Build DuckDB query for efficient loading
                cols_str = ', '.join(f'"{col}"' for col in cols_needed)
                
                if args.filter:
                    # Apply filter during scan (predicate pushdown)
                    query = f"SELECT {cols_str} FROM read_parquet('{args.input}') WHERE {args.filter}"
                    logger.info(f"Applying filter during scan: {args.filter}")
                else:
                    query = f"SELECT {cols_str} FROM read_parquet('{args.input}')"
                
                logger.debug(f"DuckDB query: {query}")
                df = duckdb.query(query).to_df()
                logger.info(f"Loaded {len(df)} rows, {len(df.columns)} columns")
                
            except Exception as e:
                logger.error(f"DuckDB query failed: {e}")
                logger.error("Falling back to pandas")
                use_duckdb = False
        
        if not use_duckdb:
            # Fallback: Load with pandas/dask (but still only needed columns)
            logger.info(f"Loading data with {'Dask' if args.use_dask else 'pandas'}")
            logger.info(f"Columns to load: {cols_needed}")
            
            if args.use_dask:
                npartitions = args.dask_npartitions or max(1, (os.cpu_count() or 2) - 1)
                logger.info(f"Using {npartitions} Dask partitions")
                
                try:
                    ddf = dd.read_parquet(args.input, engine='pyarrow', columns=cols_needed)
                    if ddf.npartitions != npartitions:
                        logger.debug(f"Repartitioning from {ddf.npartitions} to {npartitions} partitions")
                        ddf = ddf.repartition(npartitions=npartitions)
                except Exception as e:
                    logger.error(f"Failed to load with Dask: {e}")
                    sys.exit(1)
                
                # Apply filter if specified
                if args.filter:
                    logger.info(f"Applying filter: {args.filter}")
                    try:
                        ddf = ddf.query(args.filter)
                    except Exception as e:
                        logger.error(f"Filter failed: {e}")
                        sys.exit(1)
                
                logger.info("Computing to pandas DataFrame...")
                df = ddf.compute()
                logger.info(f"Loaded {len(df)} rows")
            else:
                try:
                    df = pd.read_parquet(args.input, columns=cols_needed)
                    logger.info(f"Loaded {len(df)} rows")
                except Exception as e:
                    logger.error(f"Failed to load with pandas: {e}")
                    sys.exit(1)
                
                # Apply filter if specified
                if args.filter:
                    logger.info(f"Applying filter: {args.filter}")
                    try:
                        df = df.query(args.filter)
                        logger.info(f"After filtering: {len(df)} rows")
                    except Exception as e:
                        logger.error(f"Filter failed: {e}")
                        sys.exit(1)
        
        # Load sidecar (always use pandas for small sidecar files)
        logger.info(f"Loading sidecar from {sidecar_path}")
        sidecar = pd.read_parquet(sidecar_path)
        logger.info(f"Sidecar contains {len(sidecar)} mappings")
        
        # Perform aggregation
        logger.info("Starting aggregation")
        agg_result = aggregate_by_sidecar(
            original=df,
            sidecar=sidecar,
            value_columns=args.columns,
            aggs=args.aggs,
            min_count=args.min_count,
        )
        
        logger.info(f"Aggregation complete: {len(agg_result)} HEALPix cells")
        
        # Apply densification if requested
        if args.densify:
            logger.info("Densifying output to full HEALPix grid")
            nside_value = sidecars_df.iloc[args.sidecar_index].get('nside')
            if nside_value is None or pd.isna(nside_value):
                logger.error("Cannot densify: nside not found in sidecar metadata")
                sys.exit(1)
            agg_result = densify_healpix_aggregates(agg_result, int(nside_value))
        
        # Save output
        logger.info(f"Writing output to {args.output}")
        
        # Add metadata to output
        sidecar_metadata = sidecars_df.iloc[args.sidecar_index].to_dict()
        output_metadata = {
            'source_file': str(args.input),
            'sidecar_file': str(sidecar_path),
            'healpix_mode': sidecar_metadata.get('mode', 'unknown'),
            'healpix_nside': str(sidecar_metadata.get('nside', '')),
            'healpix_order': sidecar_metadata.get('order', 'unknown'),
            'value_columns': ','.join(args.columns),
            'aggregations': ','.join(args.aggs or ['mean', 'median', 'std', 'robust_std']),
            'filter_query': args.filter or '',
            'min_count': str(args.min_count),
            'densified': str(args.densify),
            'used_duckdb': str(use_duckdb),
            'used_dask': str(args.use_dask),
            'dask_npartitions': str(args.dask_npartitions) if args.use_dask else '',
            'columns_loaded': ','.join(cols_needed),
            'output_shape': f"{agg_result.shape[0]}x{agg_result.shape[1]}",
        }
        
        agg_result.to_parquet(args.output, index=True)
        
        # Write metadata as a separate JSON sidecar
        metadata_path = args.output.with_suffix('.meta.json')
        import json
        with open(metadata_path, 'w') as f:
            json.dump(output_metadata, f, indent=2)
        
        logger.info(f"Wrote metadata to {metadata_path}")
        logger.info("Aggregation complete!")
    
    # If no action specified, show help
    if not (args.schema or args.list_sidecars or args.aggregate or args.sidecar_schema is not None):
        parser.print_help()
        sys.exit(0)
    
    logger.info("Done!")


if __name__ == '__main__':
    main()



## Usage Example

See the `main()` function for CLI usage, or import functions directly for programmatic use.